In [1]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

GPU: NVIDIA L40S
CWD: /home/j-j14d208/2_ml_pipeline


In [2]:
!pip install lightning lightgbm --quiet

In [3]:
"""Stacking 앙상블 백테스트
Walk-Forward 체크포인트를 사용하여 시간순 Stacking 백테스트.

전략:
  1. 각 3개월 윈도우의 TFT+LightGBM 체크포인트 로드
  2. 이전 윈도우 예측으로 메타모델(Stacking) 학습
  3. 현재 윈도우 예측으로 매수/매도 신호 생성
  4. Top-N 포트폴리오 시뮬레이션
"""
import warnings, json, joblib
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__}")

PyTorch: 2.10.0+cu128


In [4]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
RAW_OHLCV_PATH = BASE_PATH / "raw" / "ohlcv"
RAW_MACRO_PATH = BASE_PATH / "raw" / "macro"
MODEL_SAVE_PATH = BASE_PATH / "models" / "ensemble_backtest"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

# 체크포인트 경로
TFT_CKPT_DIR = BASE_PATH / "models" / "tft_v2_wf" / "checkpoints"
LGBM_CKPT_DIR = BASE_PATH / "models" / "lgbm_wf" / "checkpoints"
GARCH_CKPT_DIR = BASE_PATH / "models" / "garch_wf" / "checkpoints"

# 백테스트 기간: 2021 Q2 부터 (Q1은 메타모델 학습용)
BT_START_WINDOW = 1   # Window 1 (2021 Q2)부터 백테스트
MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 256

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

# 포트폴리오 설정
INITIAL_CAPITAL = 100_000_000
BUY_COMMISSION = 0.00015
SELL_COMMISSION = 0.00215
TOP_N = 10
STOP_LOSS = -0.03
TAKE_PROFIT = 0.07
MODEL_EXIT = 0.45
MAX_HOLD_DAYS = 5
MAX_POSITIONS = 20

# Walk-Forward 윈도우
WF_START = "2021-01-01"
WF_END = "2026-01-31"
WF_STEP_MONTHS = 3

windows = []
current = pd.Timestamp(WF_START)
end = pd.Timestamp(WF_END)
while current < end:
    test_end = current + pd.DateOffset(months=WF_STEP_MONTHS) - pd.Timedelta(days=1)
    if test_end > end: test_end = end
    train_end = current - pd.Timedelta(days=1)
    windows.append((str(train_end.date()), str(current.date()), str(test_end.date())))
    current += pd.DateOffset(months=WF_STEP_MONTHS)

print(f"윈도우: {len(windows)}개, 백테스트: Window {BT_START_WINDOW}~{len(windows)-1}")
print(f"전략: Top-{TOP_N}, 손절 {STOP_LOSS*100}%, 익절 +{TAKE_PROFIT*100}%, {MAX_HOLD_DAYS}일")

윈도우: 21개, 백테스트: Window 1~20
전략: Top-10, 손절 -3.0%, 익절 +7.000000000000001%, 5일


In [5]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
N_FEATURES = len(CLEANED_FEATURES)
print(f"데이터: {len(df):,} rows, {N_FEATURES} features")

데이터: 884,602 rows, 8 features


In [6]:
# ===== TFT v2 모델 정의 (로드용) =====
class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc=nn.Linear(d,d); self.gate=nn.Linear(d,d)
    def forward(self, x): return self.fc(x)*torch.sigmoid(self.gate(x))

class GRN(nn.Module):
    def __init__(self, d_in, d_h, d_out, drop=0.1):
        super().__init__()
        self.fc1=nn.Linear(d_in,d_h); self.fc2=nn.Linear(d_h,d_out)
        self.gate=GatedLinearUnit(d_out); self.norm=nn.LayerNorm(d_out)
        self.drop=nn.Dropout(drop); self.skip=nn.Linear(d_in,d_out) if d_in!=d_out else nn.Identity()
    def forward(self, x):
        r=self.skip(x); h=self.drop(F.elu(self.fc2(F.elu(self.fc1(x)))))
        return self.norm(self.gate(h)+r)

class VSN(nn.Module):
    def __init__(self, n_v, d, drop=0.1):
        super().__init__()
        self.grns=nn.ModuleList([GRN(d,d,d,drop) for _ in range(n_v)])
        self.sg=GRN(n_v*d,d,n_v,drop)
    def forward(self, x):
        B,S,V,D=x.shape
        p=torch.stack([self.grns[i](x[:,:,i,:]) for i in range(V)],dim=2)
        w=torch.softmax(self.sg(x.reshape(B,S,V*D)),dim=-1).unsqueeze(-1)
        return (p*w).sum(dim=2)

class TFTv2(pl.LightningModule):
    def __init__(self, n_feat, seq_len=30, d=128, heads=4, n_lstm=1, drop=0.2, n_cls=2, lr=5e-4, cw=None):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"]); self.lr=lr
        self.fe=nn.Linear(1,d); self.vsn=VSN(n_feat,d,drop)
        self.lstm=nn.LSTM(d,d,n_lstm,batch_first=True)
        self.attn=nn.MultiheadAttention(d,heads,dropout=drop,batch_first=True)
        self.an=nn.LayerNorm(d); self.ag=GatedLinearUnit(d)
        self.go=GRN(d,d,d,drop); self.head=nn.Linear(d,n_cls)
        self.loss_fn=nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()
    def forward(self, x):
        B,S,F=x.shape; x=self.fe(x.unsqueeze(-1)); x=self.vsn(x)
        x,_=self.lstm(x); a,_=self.attn(x,x,x); x=self.an(x+self.ag(a))
        return self.head(self.go(x[:,-1,:]))
    def training_step(self,b,_): return self.loss_fn(self(b[0]),b[1])
    def configure_optimizers(self): return torch.optim.AdamW(self.parameters(),lr=self.lr)

print("TFT v2 정의 완료")

TFT v2 정의 완료


In [7]:
# ===== 유틸 함수 =====
def make_tft_samples(full_df, start, end, seq_len, feats):
    samples, metas = [], []
    s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values; tk=g["ticker"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e:
                samples.append((v[i-seq_len:i],t[i]))
                metas.append({"date": str(d[i])[:10], "ticker": str(tk[i])})
    return samples, metas

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

def predict_tft(model, samples):
    loader = DataLoader(SeqDS(samples), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
    ps, ls = [], []
    model.eval(); model.cuda()
    with torch.no_grad():
        for x,y in loader:
            ps.extend(torch.softmax(model(x.cuda()),dim=-1)[:,1].cpu().numpy()); ls.extend(y.numpy())
    return np.array(ps), np.array(ls)

def predict_lgbm(model, full_df, start, end, feats):
    s=pd.Timestamp(start); e=pd.Timestamp(end)
    mask = (full_df["date"]>=s) & (full_df["date"]<=e)
    sub = full_df[mask]
    X = sub[feats].values.astype(np.float32)
    y = sub["target_5d"].values
    probs = model.predict_proba(X)[:, 1]
    metas = [{"date": str(d)[:10], "ticker": t} for d, t in zip(sub["date"].values, sub["ticker"].values)]
    return probs, y, metas

print("유틸 정의 완료")

유틸 정의 완료


In [8]:
# ===== Step 1: 모든 윈도우의 예측 수집 =====
# 각 윈도우별로 TFT+LightGBM 예측을 모아서 하나의 DataFrame 생성

all_predictions = []

for i, (train_end, test_start, test_end) in enumerate(windows):
    tft_ckpt = TFT_CKPT_DIR / f"window_{i:02d}_te_{train_end}.ckpt"
    lgbm_ckpt = LGBM_CKPT_DIR / f"window_{i:02d}_te_{train_end}.joblib"
    garch_ckpt = GARCH_CKPT_DIR / f"window_{i:02d}_te_{train_end}.parquet"
    
    if not tft_ckpt.exists() or not lgbm_ckpt.exists():
        print(f"  [{i:2d}] SKIP: 체크포인트 없음")
        continue
    
    print(f"  [{i:2d}] {test_start}~{test_end} 예측 생성...")
    
    # TFT 예측
    tft_model = TFTv2.load_from_checkpoint(str(tft_ckpt), strict=False)
    tft_samples, tft_metas = make_tft_samples(df, test_start, test_end, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
    if len(tft_samples) < 10:
        del tft_model; torch.cuda.empty_cache(); continue
    tft_probs, tft_labels = predict_tft(tft_model, tft_samples)
    del tft_model; torch.cuda.empty_cache()
    
    # LightGBM 예측
    lgbm_model = joblib.load(str(lgbm_ckpt))
    lgbm_probs, _, lgbm_metas = predict_lgbm(lgbm_model, df, test_start, test_end, CLEANED_FEATURES)
    del lgbm_model
    
    # GARCH
    garch_df = pd.read_parquet(str(garch_ckpt)) if garch_ckpt.exists() else None
    
    # 매칭
    tft_df = pd.DataFrame(tft_metas); tft_df["tft_prob"] = tft_probs; tft_df["label"] = tft_labels
    lgbm_df = pd.DataFrame(lgbm_metas); lgbm_df["lgbm_prob"] = lgbm_probs
    merged = tft_df.merge(lgbm_df, on=["date","ticker"], how="inner")
    merged["window"] = i
    
    if garch_df is not None and "ticker" in garch_df.columns:
        merged = merged.merge(garch_df[["ticker","vol_5d","risk_flag"]], on="ticker", how="left")
        merged["vol_5d"] = merged["vol_5d"].fillna(40.0)
        merged["risk_flag"] = merged["risk_flag"].fillna(0).astype(int)
    else:
        merged["vol_5d"] = 40.0; merged["risk_flag"] = 0
    
    all_predictions.append(merged)
    print(f"       {len(merged):,} 예측")

pred_df = pd.concat(all_predictions, ignore_index=True)
print(f"\n전체 예측: {len(pred_df):,} rows, 윈도우 {pred_df['window'].nunique()}개")

  [ 0] 2021-01-01~2021-03-31 예측 생성...
       17,931 예측
  [ 1] 2021-04-01~2021-06-30 예측 생성...
       18,916 예측
  [ 2] 2021-07-01~2021-09-30 예측 생성...
       18,808 예측
  [ 3] 2021-10-01~2021-12-31 예측 생성...
       19,397 예측
  [ 4] 2022-01-01~2022-03-31 예측 생성...
       18,426 예측
  [ 5] 2022-04-01~2022-06-30 예측 생성...
       19,440 예측
  [ 6] 2022-07-01~2022-09-30 예측 생성...
       19,719 예측
  [ 7] 2022-10-01~2022-12-31 예측 생성...
       19,382 예측
  [ 8] 2023-01-01~2023-03-31 예측 생성...
       19,304 예측
  [ 9] 2023-04-01~2023-06-30 예측 생성...
       18,928 예측
  [10] 2023-07-01~2023-09-30 예측 생성...
       19,392 예측
  [11] 2023-10-01~2023-12-31 예측 생성...
       18,835 예측
  [12] 2024-01-01~2024-03-31 예측 생성...
       19,201 예측
  [13] 2024-04-01~2024-06-30 예측 생성...
       18,854 예측
  [14] 2024-07-01~2024-09-30 예측 생성...
       19,474 예측
  [15] 2024-10-01~2024-12-31 예측 생성...
       19,252 예측
  [16] 2025-01-01~2025-03-31 예측 생성...
       18,373 예측
  [17] 2025-04-01~2025-06-30 예측 생성...
       18,978 예측
  [18] 202

In [9]:
# ===== Step 2: 롤링 Stacking 메타모델로 앙상블 확률 생성 =====
# 각 윈도우 i에서: 이전 윈도우들(0~i-1)의 예측으로 메타모델 학습 → 윈도우 i 예측

pred_df["ensemble_prob"] = np.nan

for i in range(BT_START_WINDOW, pred_df["window"].max() + 1):
    # 이전 윈도우들로 메타모델 학습
    train_mask = pred_df["window"] < i
    test_mask = pred_df["window"] == i
    
    if train_mask.sum() < 100 or test_mask.sum() == 0:
        continue
    
    X_train = pred_df[train_mask][["tft_prob", "lgbm_prob"]].values
    y_train = pred_df[train_mask]["label"].values
    X_test = pred_df[test_mask][["tft_prob", "lgbm_prob"]].values
    
    meta = LogisticRegression(random_state=42, class_weight="balanced")
    meta.fit(X_train, y_train)
    
    ensemble_probs = meta.predict_proba(X_test)[:, 1]
    pred_df.loc[test_mask, "ensemble_prob"] = ensemble_probs
    
    da = accuracy_score(pred_df[test_mask]["label"], (ensemble_probs >= 0.5).astype(int))
    print(f"  Window [{i:2d}] meta학습={train_mask.sum():,} → 예측={test_mask.sum():,} DA={da*100:.1f}% "
          f"(TFT w={meta.coef_[0][0]:.2f}, LGBM w={meta.coef_[0][1]:.2f})")

# 앙상블 확률이 없는 행 제거
bt_df = pred_df.dropna(subset=["ensemble_prob"]).copy()
print(f"\n백테스트 대상: {len(bt_df):,} rows ({bt_df['date'].min()} ~ {bt_df['date'].max()})")

  Window [ 1] meta학습=17,931 → 예측=18,916 DA=51.4% (TFT w=0.22, LGBM w=2.07)
  Window [ 2] meta학습=36,847 → 예측=18,808 DA=53.6% (TFT w=0.13, LGBM w=1.82)
  Window [ 3] meta학습=55,655 → 예측=19,397 DA=53.7% (TFT w=0.14, LGBM w=1.97)
  Window [ 4] meta학습=75,052 → 예측=18,426 DA=56.9% (TFT w=0.06, LGBM w=1.95)
  Window [ 5] meta학습=93,478 → 예측=19,440 DA=53.9% (TFT w=-0.06, LGBM w=2.42)
  Window [ 6] meta학습=112,918 → 예측=19,719 DA=52.7% (TFT w=-0.09, LGBM w=2.16)
  Window [ 7] meta학습=132,637 → 예측=19,382 DA=41.7% (TFT w=-0.39, LGBM w=2.02)
  Window [ 8] meta학습=152,019 → 예측=19,304 DA=56.6% (TFT w=-0.17, LGBM w=1.31)
  Window [ 9] meta학습=171,323 → 예측=18,928 DA=54.8% (TFT w=-0.41, LGBM w=1.39)
  Window [10] meta학습=190,251 → 예측=19,392 DA=57.5% (TFT w=-0.38, LGBM w=1.45)
  Window [11] meta학습=209,643 → 예측=18,835 DA=49.9% (TFT w=-0.48, LGBM w=1.52)
  Window [12] meta학습=228,478 → 예측=19,201 DA=52.2% (TFT w=-0.34, LGBM w=1.53)
  Window [13] meta학습=247,679 → 예측=18,854 DA=55.1% (TFT w=-0.27, LGBM w=1.40)
  Window

In [10]:
# ===== Step 3: 날짜별 예측 dict 생성 =====
predictions = {}
for _, row in bt_df.iterrows():
    d, t = row["date"], row["ticker"]
    if d not in predictions: predictions[d] = {}
    predictions[d][t] = float(row["ensemble_prob"])

print(f"예측 거래일: {len(predictions)}, 기간: {min(predictions.keys())} ~ {max(predictions.keys())}")

예측 거래일: 1186, 기간: 2021-04-01 ~ 2026-01-30


In [11]:
# ===== Step 4: 종가 데이터 로드 =====
tickers_needed = set()
for dp in predictions.values(): tickers_needed.update(dp.keys())

close_prices = {}
for ticker in tickers_needed:
    path = RAW_OHLCV_PATH / f"{ticker}.parquet"
    if not path.exists(): continue
    try:
        ohlcv = pd.read_parquet(str(path))
        ohlcv.index = pd.to_datetime(ohlcv.index)
        close_prices[ticker] = ohlcv["close"].sort_index()
    except: pass

kospi = pd.Series(dtype=float)
kospi_path = RAW_MACRO_PATH / "kospi200.parquet"
if kospi_path.exists():
    kdf = pd.read_parquet(str(kospi_path))
    kdf.index = pd.to_datetime(kdf.index)
    col = "close" if "close" in kdf.columns else kdf.columns[0]
    kospi = kdf[col].sort_index()

print(f"종가: {len(close_prices)} 종목, KOSPI200: {len(kospi)} rows")

종가: 324 종목, KOSPI200: 2730 rows


In [12]:
# ===== Step 5: 포트폴리오 시뮬레이션 =====
@dataclass
class Position:
    ticker: str; buy_date: str; buy_price: float; shares: int
    buy_prob: float; invested: float; hold_days: int = 0

def get_close(ticker, date):
    if ticker not in close_prices: return None
    s = close_prices[ticker]; ts = pd.Timestamp(date)
    if ts in s.index: return float(s.loc[ts])
    mask = s.index <= ts
    return float(s.loc[mask].iloc[-1]) if mask.any() else None

cash = INITIAL_CAPITAL
positions = []
trade_log = []
snapshots = []

trading_dates = sorted(predictions.keys())
print(f"시뮬레이션: {len(trading_dates)} 거래일")

for i, date in enumerate(trading_dates):
    for p in positions: p.hold_days += 1
    
    # 매도 체크
    date_preds = predictions.get(date, {})
    sells = []
    for pos in positions:
        price = get_close(pos.ticker, date)
        if price is None: continue
        ret = (price - pos.buy_price) / pos.buy_price
        if ret <= STOP_LOSS:
            sells.append((pos, f"stop_loss ({ret*100:.1f}%)")); continue
        cp = date_preds.get(pos.ticker)
        if cp is not None and cp < MODEL_EXIT:
            sells.append((pos, f"model_exit (p={cp:.3f})")); continue
        if ret >= TAKE_PROFIT:
            sells.append((pos, f"take_profit ({ret*100:.1f}%)")); continue
        if pos.hold_days >= MAX_HOLD_DAYS:
            sells.append((pos, f"timeout ({pos.hold_days}d)"))
    
    for pos, reason in sells:
        price = get_close(pos.ticker, date)
        if price is None: continue
        rev = pos.shares * price; comm = rev * SELL_COMMISSION; net = rev - comm
        cash += net; positions.remove(pos)
        trade_log.append({"date":date,"ticker":pos.ticker,"action":"SELL","price":price,
            "shares":pos.shares,"amount":net,"commission":comm,"pnl":net-pos.invested,
            "return":(net-pos.invested)/pos.invested,"hold_days":pos.hold_days,"reason":reason})
    
    # 매수
    preds = predictions[date]
    held = {p.ticker for p in positions}
    candidates = [(t,p) for t,p in preds.items() if t not in held and p > 0.5]
    candidates.sort(key=lambda x: -x[1])
    buys = candidates[:TOP_N]
    
    for ticker, prob in buys:
        if len(positions) >= MAX_POSITIONS: break
        price = get_close(ticker, date)
        if price is None or price <= 0: continue
        slots = MAX_POSITIONS - len(positions)
        alloc = min(cash / max(slots, 1), cash)
        shares = int(alloc / (price * (1 + BUY_COMMISSION)))
        if shares <= 0: continue
        cost = shares * price; comm = cost * BUY_COMMISSION; total = cost + comm
        if total > cash: continue
        cash -= total
        positions.append(Position(ticker, date, price, shares, prob, total))
        trade_log.append({"date":date,"ticker":ticker,"action":"BUY","price":price,
            "shares":shares,"amount":total,"commission":comm,"prob":prob})
    
    pv = cash + sum(p.shares * (get_close(p.ticker, date) or 0) for p in positions)
    snapshots.append({"date":date,"portfolio_value":pv,"cash":cash,"n_positions":len(positions)})
    
    if (i+1) % 100 == 0:
        print(f"  Day {i+1}/{len(trading_dates)} | {date} | 자산: {pv:,.0f} | 수익: {(pv/INITIAL_CAPITAL-1)*100:+.2f}%")

# 잔여 청산
if positions:
    last = trading_dates[-1]
    print(f"잔여 {len(positions)} 포지션 청산")
    for pos in list(positions):
        price = get_close(pos.ticker, last)
        if price is None: continue
        rev = pos.shares * price; comm = rev * SELL_COMMISSION; net = rev - comm
        cash += net; positions.remove(pos)
        trade_log.append({"date":last,"ticker":pos.ticker,"action":"SELL","price":price,
            "shares":pos.shares,"amount":net,"commission":comm,"pnl":net-pos.invested,
            "return":(net-pos.invested)/pos.invested,"hold_days":pos.hold_days,"reason":"backtest_end"})

print(f"시뮬레이션 완료")

시뮬레이션: 1186 거래일
  Day 100/1186 | 2021-08-23 | 자산: 106,658,492 | 수익: +6.66%
  Day 200/1186 | 2022-01-18 | 자산: 108,320,026 | 수익: +8.32%
  Day 300/1186 | 2022-06-17 | 자산: 115,449,986 | 수익: +15.45%
  Day 400/1186 | 2022-11-11 | 자산: 116,670,187 | 수익: +16.67%
  Day 500/1186 | 2023-04-06 | 자산: 117,871,774 | 수익: +17.87%
  Day 600/1186 | 2023-08-31 | 자산: 124,596,755 | 수익: +24.60%
  Day 700/1186 | 2024-01-30 | 자산: 108,671,372 | 수익: +8.67%
  Day 800/1186 | 2024-06-28 | 자산: 123,404,254 | 수익: +23.40%
  Day 900/1186 | 2024-11-26 | 자산: 121,051,831 | 수익: +21.05%
  Day 1000/1186 | 2025-04-25 | 자산: 129,915,146 | 수익: +29.92%
  Day 1100/1186 | 2025-09-22 | 자산: 138,683,680 | 수익: +38.68%
잔여 11 포지션 청산
시뮬레이션 완료


In [13]:
# ===== Step 6: 결과 계산 =====
snap_df = pd.DataFrame(snapshots)
snap_df["date"] = pd.to_datetime(snap_df["date"])
trade_df = pd.DataFrame(trade_log)
sell_df = trade_df[trade_df["action"]=="SELL"] if len(trade_df) > 0 else pd.DataFrame()

final_value = snap_df["portfolio_value"].iloc[-1]
total_return = final_value / INITIAL_CAPITAL - 1
n_days = len(snap_df)
annual_return = (1 + total_return) ** (252 / max(n_days, 1)) - 1

daily_ret = snap_df["portfolio_value"].pct_change().dropna()
sharpe = float(daily_ret.mean() / daily_ret.std() * np.sqrt(252)) if daily_ret.std() > 1e-9 else 0
cummax = snap_df["portfolio_value"].cummax()
mdd = float(((snap_df["portfolio_value"] - cummax) / cummax).min())

if len(sell_df) > 0:
    win_rate = (sell_df["pnl"] > 0).mean()
    total_comm = trade_df["commission"].sum()
    avg_win = sell_df[sell_df["pnl"]>0]["return"].mean() if (sell_df["pnl"]>0).any() else 0
    avg_loss = sell_df[sell_df["pnl"]<=0]["return"].mean() if (sell_df["pnl"]<=0).any() else 0
else:
    win_rate = total_comm = avg_win = avg_loss = 0

# 벤치마크
bm_ret = None
bt_s = pd.Timestamp(snap_df["date"].iloc[0])
bt_e = snap_df["date"].iloc[-1]
bm_slice = kospi[(kospi.index >= bt_s) & (kospi.index <= bt_e)]
if len(bm_slice) >= 2:
    bm_ret = float(bm_slice.iloc[-1] / bm_slice.iloc[0] - 1)

print("="*70)
print("  Stacking 앙상블 포트폴리오 백테스트 결과")
print("="*70)
print(f"  기간: {snap_df['date'].iloc[0].date()} ~ {snap_df['date'].iloc[-1].date()} ({n_days}일)")
print(f"  초기 자본: {INITIAL_CAPITAL:>15,}원")
print(f"  최종 자산: {final_value:>15,.0f}원")
print(f"  총 수익률: {total_return*100:>+8.2f}%")
print(f"  연환산:    {annual_return*100:>+8.2f}%")
print(f"  Sharpe:    {sharpe:>8.3f}")
print(f"  MDD:       {mdd*100:>8.2f}%")
if bm_ret is not None:
    print(f"  KOSPI200:  {bm_ret*100:>+8.2f}%")
    print(f"  초과수익:  {(total_return-bm_ret)*100:>+8.2f}%")
print(f"  총 거래:   {len(sell_df):>8}건")
print(f"  승률:      {win_rate*100:>8.1f}%")
print(f"  평균이익:  {avg_win*100:>+8.2f}%")
print(f"  평균손실:  {avg_loss*100:>+8.2f}%")
print(f"  총수수료:  {total_comm:>12,.0f}원")
print()
if len(sell_df) > 0:
    reasons = sell_df["reason"].apply(lambda x: x.split(" ")[0]).value_counts()
    print("  [매도 사유]")
    for r, c in reasons.items():
        print(f"    {r:16s}: {c:5d}건 ({c/len(sell_df)*100:.1f}%)")
print("="*70)
print(f"\n비교:")
print(f"  LSTM Top-10:     수익률=+48.95%, Sharpe=1.974, MDD=-17.75%")
print(f"  TFT v2 Top-10:   수익률=+34.92%, Sharpe=1.567, MDD=-12.10%")

  Stacking 앙상블 포트폴리오 백테스트 결과
  기간: 2021-04-01 ~ 2026-01-30 (1186일)
  초기 자본:     100,000,000원
  최종 자산:     147,970,050원
  총 수익률:   +47.97%
  연환산:       +8.68%
  Sharpe:       0.528
  MDD:         -18.42%
  KOSPI200:    +83.21%
  초과수익:    -35.24%
  총 거래:       5299건
  승률:          46.1%
  평균이익:     +4.46%
  평균손실:     -3.48%
  총수수료:    73,017,581원

  [매도 사유]
    timeout         :  1769건 (33.4%)
    stop_loss       :  1700건 (32.1%)
    model_exit      :  1213건 (22.9%)
    take_profit     :   606건 (11.4%)
    backtest_end    :    11건 (0.2%)

비교:
  LSTM Top-10:     수익률=+48.95%, Sharpe=1.974, MDD=-17.75%
  TFT v2 Top-10:   수익률=+34.92%, Sharpe=1.567, MDD=-12.10%


In [14]:
# ===== 저장 =====
snap_df.to_parquet(str(MODEL_SAVE_PATH / f"equity_curve_{datetime.now().strftime('%Y%m%d')}.parquet"))
if len(trade_df) > 0:
    trade_df.to_parquet(str(MODEL_SAVE_PATH / f"trade_log_{datetime.now().strftime('%Y%m%d')}.parquet"))

summary = {"model": "Stacking_Ensemble",
  "period": f"{snap_df['date'].iloc[0].date()}~{snap_df['date'].iloc[-1].date()}",
  "total_return": round(total_return*100, 2), "annual_return": round(annual_return*100, 2),
  "sharpe": round(sharpe, 3), "mdd": round(mdd*100, 2),
  "win_rate": round(win_rate*100, 1), "total_trades": len(sell_df),
  "total_commission": round(total_comm),
  "benchmark": round(bm_ret*100, 2) if bm_ret else None}
json.dump(summary, open(str(MODEL_SAVE_PATH/"backtest_summary.json"),"w"), indent=2)
print(f"저장: {MODEL_SAVE_PATH}")

저장: /home/j-j14d208/kospi200-project/models/ensemble_backtest


## Stacking 앙상블 백테스트\n\n### 핵심 구조\n```\nWindow 0: TFT+LGBM 예측 수집 (메타모델 학습용)\nWindow 1: Window 0으로 메타모델 학습 -> Window 1 예측 -> 매수/매도\nWindow 2: Window 0-1로 메타모델 학습 -> Window 2 예측 -> 매수/매도\n...\nWindow 20: Window 0-19로 메타모델 학습 -> Window 20 예측 -> 매수/매도\n```\n→ 시간순 메타모델 (미래 정보 누수 없음)\n\n### 비교\n| 모델 | 수익률 | Sharpe | MDD |\n|------|--------|--------|-----|\n| LSTM Top-10 | +48.95% | 1.974 | -17.75% |\n| TFT v2 Top-10 | +34.92% | 1.567 | -12.10% |\n| **Stacking 앙상블** | **?%** | **?** | **?%** |